# Pretrained Audio Embeddings — MusicNN · Dual-GPU Harness

**Strategy: data-parallel sharding across 2 GPUs**

MusicNN uses TensorFlow, which builds a global compute graph per process.
Sharing one TF session across threads causes session lock contention — it's
actually slower than serial.  The correct approach is **one process per GPU**:

```
  File list (400k tracks)
      ├── shard 0 (tracks 0…199k)  →  subprocess 0  (CUDA_VISIBLE_DEVICES=0)
      └── shard 1 (tracks 200k…)  →  subprocess 1  (CUDA_VISIBLE_DEVICES=1)
            ↓                               ↓
    checkpoints_gpu0/          checkpoints_gpu1/
            └──────────────────────────────┘
                       merge
                  musicnn_embeddings_400k.parquet
```

Each subprocess has its own TF session, its own GPU, zero contention.
Near-linear 2× speedup over single-GPU.


In [1]:
# %pip install tensorflow tf-keras
# %pip install musicnn --no-deps


In [2]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import os, sys, glob, time, logging, subprocess, json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)


## Configuration

In [3]:
PREVIEWS_DIR    = "/home/build/martin/spotify_v2/data/previews"
CHECKPOINT_BASE = "./checkpoints_musicnn"   # sub-dirs _gpu0 / _gpu1 created automatically
OUTPUT_PARQUET  = "./musicnn_embeddings_400k.parquet"

MODEL           = "MSD_musicnn"    # or MSD_musicnn_big (richer, slower)
LAYER           = "penultimate"    # 200-d semantic embedding
INPUT_LENGTH    = 3                # seconds per patch
INPUT_OVERLAP   = True
L2_NORMALISE    = True
PROCS_PER_GPU   = 8

CHECKPOINT_N    = 200            # flush every N tracks per worker

# GPU IDs — set to [0, 1] for two GPUs, or [0] to run on a single GPU
GPU_IDS         = [0, 1]


## Worker script (written to disk, executed as subprocess)

In [4]:
WORKER_SCRIPT = '''
import os, sys, json, time, logging, warnings, contextlib
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow as tf

# Critical for multi-process GPU sharing
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e: print(e)

from musicnn.extractor import extractor
warnings.filterwarnings("ignore")

gpu_id = sys.argv[4] if len(sys.argv) > 4 else "?"
logging.basicConfig(
    level=logging.INFO,
    format=f"%(asctime)s [GPU {gpu_id}] %(levelname)s %(message)s",
)
log = logging.getLogger(__name__)

def load_processed_ids(checkpoint_dir):
    done = set()
    for p in Path(checkpoint_dir).glob("checkpoint_*.parquet"):
        try:
            done.update(pd.read_parquet(p, columns=["track_id"])["track_id"].tolist())
        except Exception: pass
    return done

def flush(buffer, checkpoint_dir, idx):
    if not buffer: return
    ids   = [r[0] for r in buffer]
    feats = np.vstack([r[1] for r in buffer])
    df    = pd.DataFrame(feats, columns=[f"e{i}" for i in range(feats.shape[1])])
    df.insert(0, "track_id", ids)
    path  = Path(checkpoint_dir) / f"checkpoint_{idx:05d}.parquet"
    df.to_parquet(path, index=False, compression="snappy")
    print(f"[CHECKPOINT] saved {path.absolute()}")

def extract(file_path, model, layer, input_length, input_overlap, l2):
    track_id = Path(file_path).stem
    try:
        with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):
            _, _, features = extractor(file_path, model=model, input_length=input_length, input_overlap=input_overlap, extract_features=True)
        emb = np.mean(features[layer], axis=0).astype(np.float32)
        if np.isnan(emb).any(): return track_id, None
        if l2:
            n = np.linalg.norm(emb)
            if n > 0: emb = emb / n
        return track_id, emb
    except Exception: return track_id, None

if __name__ == "__main__":
    files_json, checkpoint_dir, params_json, gpu_id = sys.argv[1:5]
    todo   = json.loads(open(files_json).read())
    params = json.loads(params_json)
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    done_ids = load_processed_ids(checkpoint_dir)
    todo     = [f for f in todo if Path(f).stem not in done_ids]
    print(f"[INFO] {len(todo)} tracks to process in this worker")
    buffer, ckpt_i = [], len(list(Path(checkpoint_dir).glob("checkpoint_*.parquet")))
    for fp in todo:
        track_id, emb = extract(fp, params["model"], params["layer"], params["input_length"], params["input_overlap"], params["l2"])
        if emb is not None: buffer.append((track_id, emb))
        else: print(f"[ERR] Failed to extract {track_id}")
        print("PROG_INC: 1")
        sys.stdout.flush()
        if len(buffer) >= params["checkpoint_n"]:
            flush(buffer, checkpoint_dir, ckpt_i)
            ckpt_i += 1; buffer.clear()
    flush(buffer, checkpoint_dir, ckpt_i)
'''

WORKER_PATH = Path("/tmp/musicnn_worker.py")
WORKER_PATH.write_text(WORKER_SCRIPT)
log.info(f"Worker script written → {WORKER_PATH}")


2026-03-20 17:37:26,936 INFO Worker script written → /tmp/musicnn_worker.py


## Verify GPUs and TF

In [5]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
log.info(f"TF {tf.__version__} sees {len(gpus)} GPU(s): {[g.name for g in gpus]}")
assert len(gpus) >= 2, f"Expected 2 GPUs, found {len(gpus)}"
for i, g in enumerate(gpus):
    log.info(f"  GPU {i}: {g.name}")


I0000 00:00:1774028247.463796  538736 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774028247.522781  538736 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774028250.211345  538736 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 17:37:30,926 INFO TF 2.21.0 sees 2 GPU(s): ['/physical_device:GPU:0', '/physical_device:GPU:1']
2026-03-20 17:37:30,929 INF

## Shard files and launch both GPU processes

In [6]:
all_files = sorted(glob.glob(os.path.join(PREVIEWS_DIR, "*.mp3")))
log.info(f"Found {len(all_files):,} total files")

done_ids = set()
for gpu_dir in Path(CHECKPOINT_BASE).glob("gpu*"):
    for p in gpu_dir.glob("checkpoint_*.parquet"):
        try: done_ids.update(pd.read_parquet(p, columns=["track_id"])["track_id"].tolist())
        except Exception: pass
log.info(f"Already processed: {len(done_ids):,} files")

remaining_files = [f for f in all_files if Path(f).stem not in done_ids]
log.info(f"Remaining to process: {len(remaining_files):,} files")

TOTAL_WORKERS = len(GPU_IDS) * PROCS_PER_GPU
shards = [all_files[i::TOTAL_WORKERS] for i in range(TOTAL_WORKERS)]

params = {
    "model": MODEL, "layer": LAYER, "input_length": INPUT_LENGTH,
    "input_overlap": INPUT_OVERLAP, "l2": L2_NORMALISE,
    "checkpoint_n": CHECKPOINT_N,
}

procs = []
for i in range(TOTAL_WORKERS):
    gpu_id = GPU_IDS[i % len(GPU_IDS)]
    worker_local_id = i // len(GPU_IDS)
    
    shard_path = Path(f"/tmp/shard_w{i}.json")
    shard_path.write_text(json.dumps(shards[i]))

    # Separate checkpoint dirs for each worker to avoid parquet write conflicts
    ckpt_dir = Path(CHECKPOINT_BASE) / f"gpu{gpu_id}_w{worker_local_id}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PYTHONUNBUFFERED"] = "1"
    env["TF_USE_LEGACY_KERAS"] = "1"

    cmd = [sys.executable, "-u", str(WORKER_PATH),
           str(shard_path), str(ckpt_dir),
           json.dumps(params), str(gpu_id)]

    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    procs.append((f"{gpu_id}:{worker_local_id}", p))
    if i % 4 == 0: log.info(f"Launched worker {i}/{TOTAL_WORKERS}...")
log.info(f"All {TOTAL_WORKERS} workers launched.")


2026-03-20 17:37:32,274 INFO Found 402,088 total files
2026-03-20 17:37:51,321 INFO Already processed: 388,061 files
2026-03-20 17:37:54,205 INFO Remaining to process: 14,027 files
2026-03-20 17:37:54,226 INFO Launched worker 0/16...
2026-03-20 17:37:54,331 INFO Launched worker 4/16...
2026-03-20 17:37:54,509 INFO Launched worker 8/16...
2026-03-20 17:37:54,724 INFO Launched worker 12/16...
2026-03-20 17:37:54,912 INFO All 16 workers launched.


## Stream logs from both processes

In [7]:
import threading, time as _time
from tqdm.auto import tqdm

pbar = tqdm(total=len(remaining_files), desc="Processing tracks", unit="track")

def stream_logs(gpu_id, proc):
    for line in proc.stdout:
        if "PROG_INC: 1" in line:
            pbar.update(1)
        elif "[CHECKPOINT]" in line or "[INFO]" in line or "[ERR]" in line:
            print(f"[GPU {gpu_id}] {line.strip()}", flush=True)
        elif "ERROR" in line or "Exception" in line:
            print(f"[GPU {gpu_id}] {line}", end="", flush=True)
        # Silent for normal progress to keep it clean

log_threads = [threading.Thread(target=stream_logs, args=(gid, p), daemon=True) for gid, p in procs]
for t in log_threads: t.start()

# Wait for both processes to finish
for gid, p in procs:
    rc = p.wait()
    if rc != 0:
        log.error(f"GPU {gid} worker failed with code {rc}")

for t in log_threads: t.join()
pbar.close()
log.info("All workers finished.")


Processing tracks:   0%|          | 0/14027 [00:00<?, ?track/s]

[GPU 0:0] [INFO] 1131 tracks to process in this worker
[GPU 0:4] [INFO] 930 tracks to process in this worker
[GPU 0:6] [INFO] 930 tracks to process in this worker
[GPU 1:3] [INFO] 931 tracks to process in this worker
[GPU 1:0] [INFO] 1131 tracks to process in this worker
[GPU 1:1] [INFO] 1131 tracks to process in this worker
[GPU 1:7] [INFO] 930 tracks to process in this worker
[GPU 1:6] [INFO] 1130 tracks to process in this worker
[GPU 0:3] [INFO] 931 tracks to process in this worker
[GPU 1:5] [INFO] 930 tracks to process in this worker
[GPU 0:7] [INFO] 1130 tracks to process in this worker
[GPU 0:1] [INFO] 931 tracks to process in this worker
[GPU 1:4] [INFO] 930 tracks to process in this worker
[GPU 0:2] [INFO] 0 tracks to process in this worker
[GPU 1:2] [INFO] 1 tracks to process in this worker
[GPU 0:0] [ERR] Failed to extract 6EDqtSdBVCIDqvnvTHgwFn
[GPU 0:5] [INFO] 930 tracks to process in this worker
[GPU 1:7] [ERR] Failed to extract 4gGIY9VXyYvAgh7D6GAuBl
[GPU 1:3] [ERR] Faile

2026-03-20 18:44:58,413 INFO All workers finished.


## Merge all GPU checkpoints → final Parquet

In [8]:
def merge_all_checkpoints(checkpoint_base: str, output_path: str) -> pd.DataFrame:
    checkpoint_base = Path(checkpoint_base)
    all_parts = []

    # Report what's on disk before attempting merge
    for gpu_dir in sorted(checkpoint_base.glob("gpu*")):
        parts = sorted(gpu_dir.glob("checkpoint_*.parquet"))
        log.info(f"  {gpu_dir.name}: {len(parts)} checkpoint file(s)")
        all_parts.extend(parts)

    if not all_parts:
        # Give a clear, actionable error
        present = list(checkpoint_base.iterdir()) if checkpoint_base.exists() else []
        raise FileNotFoundError(
            f"No checkpoint parquet files found under {checkpoint_base}.\n"
            f"Contents of {checkpoint_base}: {present}\n"
            "Possible causes:\n"
            "  • The worker processes failed — check [GPU x] log output above\n"
            "  • CHECKPOINT_BASE path doesn't match where workers wrote files\n"
            "  • Workers finished but haven't flushed yet (shouldn't happen after wait cell)\n"
            "Fix the underlying issue and re-run from the launch cell."
        )

    log.info(f"Merging {len(all_parts)} checkpoint file(s) …")
    df = pd.concat([pd.read_parquet(p) for p in all_parts], ignore_index=True)
    df.drop_duplicates(subset="track_id", inplace=True)
    df.to_parquet(output_path, index=False, compression="snappy")
    log.info(f"Final → {output_path}  ({len(df):,} rows, {df.shape[1]-1} dims)")
    return df

df_emb = merge_all_checkpoints(CHECKPOINT_BASE, OUTPUT_PARQUET)
print(df_emb.shape)
df_emb.head(3)


2026-03-20 18:45:45,023 INFO   gpu0_w0: 126 checkpoint file(s)
2026-03-20 18:45:45,132 INFO   gpu0_w1: 126 checkpoint file(s)
2026-03-20 18:45:45,133 INFO   gpu0_w2: 126 checkpoint file(s)
2026-03-20 18:45:45,133 INFO   gpu0_w3: 126 checkpoint file(s)
2026-03-20 18:45:45,134 INFO   gpu0_w4: 126 checkpoint file(s)
2026-03-20 18:45:45,135 INFO   gpu0_w5: 126 checkpoint file(s)
2026-03-20 18:45:45,136 INFO   gpu0_w6: 126 checkpoint file(s)
2026-03-20 18:45:45,136 INFO   gpu0_w7: 126 checkpoint file(s)
2026-03-20 18:45:45,137 INFO   gpu1_w0: 126 checkpoint file(s)
2026-03-20 18:45:45,138 INFO   gpu1_w1: 126 checkpoint file(s)
2026-03-20 18:45:45,139 INFO   gpu1_w2: 126 checkpoint file(s)
2026-03-20 18:45:45,140 INFO   gpu1_w3: 126 checkpoint file(s)
2026-03-20 18:45:45,140 INFO   gpu1_w4: 126 checkpoint file(s)
2026-03-20 18:45:45,141 INFO   gpu1_w5: 126 checkpoint file(s)
2026-03-20 18:45:45,142 INFO   gpu1_w6: 126 checkpoint file(s)
2026-03-20 18:45:45,143 INFO   gpu1_w7: 126 checkpoint 

(402082, 201)


,track_id,e0,e1,e2,e3,e4,e5,e6,e7,e8,...,e190,e191,e192,e193,e194,e195,e196,e197,e198,e199
0,0009mEWM7HILVo4VZYtqwc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00188mZCCzt7sKzADYWZC8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,002Z8jNRKK4JxmCnhlu36r,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Nearest-neighbour sanity check

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

QUERY_IDX = 0
N_RESULTS = 10

feat_cols = [c for c in df_emb.columns if c != "track_id"]
mat       = df_emb[feat_cols].values.astype(np.float32)
query     = mat[QUERY_IDX:QUERY_IDX+1]
sims      = cosine_similarity(query, mat)[0]

top_idx   = np.argsort(sims)[::-1][1:N_RESULTS+1]
results   = df_emb.iloc[top_idx][["track_id"]].copy()
results["similarity"] = sims[top_idx]
print(f"Query: {df_emb.iloc[QUERY_IDX]['track_id']}")
print(results.to_string(index=False))


ValueError: Input contains NaN.

## Monitoring & tuning

```bash
# In a separate terminal — watch both GPUs in real time
watch -n 0.5 nvidia-smi

# Or for a cleaner view:
nvidia-smi dmon -s u -d 1
```

| Symptom | Cause | Fix |
|---|---|---|
| One GPU at 0% | Wrong `CUDA_VISIBLE_DEVICES` | Check subprocess env; confirm TF sees GPU |
| Both GPUs < 60% | I/O bound (slow MP3 decode inside musicnn) | Pre-convert to WAV with ffmpeg |
| OOM on GPU | Large model + long clips | Use `MSD_musicnn` not `_big`; set `INPUT_LENGTH=2` |
| Process exits with rc=1 | musicnn/TF import error | Check `%pip install` cell ran successfully |

## Combining with MFCC features

```python
df_mfcc = pd.read_parquet("mfcc_features_400k_gpu.parquet")
df_full = df_emb.merge(df_mfcc, on="track_id", how="inner", suffixes=("_nn", "_mfcc"))
# L2-normalise each block independently before DL training input
```
